# Vet record timeline

**What this notebook does:** reads the veterinary records of one pet, asks Gemini to list every
visit in them, and puts the visits from all the files into one timeline ordered by date.

**How it works, in order:**

1. The record files of one pet are found in BigQuery (query to be filled in) and downloaded to
   `data/records/<pet name or claim id>/`. PDF, JPG and PNG files are read.
2. Each file is sent to Gemini through Vertex AI (Google Cloud's service for running LLMs).
3. Gemini returns one entry per visit, in a fixed format (see "The output format" below).
4. The answer for each file is saved as a JSON file in `output/`.
5. The visits from all files are joined into one table, sorted by visit date.

**Before the first run:**

- Kernel: `.venv/bin/python` in this folder (created with `poetry install --no-root`).
- `gcloud auth application-default login` (signs this machine in to Google Cloud).
- Fill in the settings in the next cell.

In [ ]:
from pathlib import Path

# --- Settings: fill these in ---

# The Google Cloud project that pays for the Gemini calls. Not confirmed: check which project to use.
PROJECT_ID = "FILL-IN"

# The Google Cloud region the calls run in. Must be in the EU (gdpr-compliance.md RULE 6).
# Not confirmed that every Gemini version is offered in this region.
LOCATION = "europe-west1"

# The Gemini versions to compare. The extraction runs once per file for each name in this list.
MODELS = ["gemini-3.7-flash"]

# The folder with one pet's record files, and the folder the answers are saved to.
PET = "FILL-IN"
RECORDS_DIR = Path("data/records") / PET
OUTPUT_DIR = Path("output") / PET

In [ ]:
import json

import pandas as pd
from google import genai
from google.genai import types
from google.cloud import bigquery
from pydantic import BaseModel

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

## BigQuery

Same set-up as `pricing-analysis/.../data_exploration_rk.ipynb`: your own Google login
(`gcloud auth application-default login`), and a helper that dry-runs every query first. A dry run
asks BigQuery how many bytes the query would read without running it; the helper stops if that is
above `max_gb` (`cost-controls.md` RULE 1).

The query that finds the pet record files goes in the cell after the helper, once the table names
are known.

In [ ]:
# BigQuery client (your own Google login) and a query helper that dry-runs first
bq_client = bigquery.Client(project='hedvig-coruscant')


def run_query(sql, max_gb=10):
    dry = bq_client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = dry.total_bytes_processed / 1e9
    print(f'Dry run: this query scans {gb:.2f} GB')
    if gb > max_gb:
        raise ValueError(f'{gb:.2f} GB is above max_gb={max_gb}. Approve by raising max_gb.')
    return bq_client.query(sql).result().to_dataframe()

In [ ]:
# The record files to fetch from AWS S3: the first patient-record upload of the first Pet claim
# per contract (ManyPets excluded), uploaded after 2025-01-01, not deleted. 30 files picked in a
# fixed pseudo-random order. Dry run: ~0.17 GB.
# Only the columns needed to fetch the files are selected (gdpr-compliance.md RULE 2).
# PII: file_name may hold a pet or person name; do not print or log it.
SQL = """
WITH first_pet_claim AS (
  SELECT contract_id, claim_id
  FROM `hedvig-coruscant.dbt_prod.claims_reporting`
  WHERE promise_category = 'Pet'
    AND acquisition_source != 'manypets'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY contract_id ORDER BY notification_date, claim_id) = 1
)
SELECT f.id, f.claims_id, f.bucket, f.key, f.file_name, f.content_type, f.uploaded_at
FROM `hedvig-coruscant.dbt_prod.stg_claims_service__claim_file` f
JOIN first_pet_claim p ON p.claim_id = f.claims_id
WHERE f.file_name LIKE '%patient%'
  AND DATE(f.uploaded_at) > '2025-01-01'
  AND f.deleted_at IS NULL
  AND f.marked_as_deleted_at IS NULL
  AND f.upload_source IN ('MANUAL', 'APP')
QUALIFY ROW_NUMBER() OVER (PARTITION BY f.claims_id ORDER BY f.uploaded_at, f.id) = 1
ORDER BY FARM_FINGERPRINT(CAST(f.id AS STRING))
LIMIT 30
"""
record_files_table = run_query(SQL)

## The output format

One entry per visit. Gemini is made to answer in exactly this shape, so the answers from
different files and different Gemini versions can be compared column by column.

| field | what it holds |
|---|---|
| `visit_date` | the date of the visit, as `YYYY-MM-DD`; empty if the record gives none |
| `clinic` | the clinic or vet, as written in the record |
| `reason_for_visit` | why the animal was brought in |
| `symptoms` | the symptoms and findings noted at the visit |
| `diagnoses` | the diagnoses given at the visit; empty list if none |
| `treatment` | what was done or prescribed |
| `vet_instructions` | what the vet told the owner to do next |
| `page` | the page of the file the visit is on |
| `source_quote` | a short quote from the record, in its original language, that the entry is based on |

`page` and `source_quote` are there so a person can check each entry against the record.

In [ ]:
class Visit(BaseModel):
    visit_date: str | None
    clinic: str | None
    reason_for_visit: str | None
    symptoms: list[str]
    diagnoses: list[str]
    treatment: str | None
    vet_instructions: str | None
    page: int | None
    source_quote: str | None


class RecordExtraction(BaseModel):
    animal_name: str | None
    species: str | None
    visits: list[Visit]

## The prompt

**This prompt is a placeholder written by Claude.** Desiree is drafting the real extraction
prompt; replace the text of `PROMPT` with it when it is ready.

In [ ]:
PROMPT = """You are reading a veterinary record for an insurance claims handler.

List every visit in the record, one entry per visit, in the order they appear.
For each visit fill in the fields of the output format.

Rules:
- Use only what is written in the record. If the record does not give a field, leave it empty.
- visit_date is the date of the visit, written as YYYY-MM-DD.
- Write symptoms, diagnoses, treatment and vet_instructions in English.
- source_quote is a short quote copied exactly from the record, in its original language.
- page is the page number of the file the visit is on, counting the first page as 1.
"""

MIME_TYPES = {".pdf": "application/pdf", ".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png"}


def extract_record(path: Path, model: str) -> RecordExtraction:
    """Send one record file to one Gemini version and return the visits it lists."""
    response = client.models.generate_content(
        model=model,
        contents=[
            types.Part.from_bytes(data=path.read_bytes(), mime_type=MIME_TYPES[path.suffix.lower()]),
            PROMPT,
        ],
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=RecordExtraction,
        ),
    )
    return response.parsed

## Run the extraction

One Gemini call per file per Gemini version. Each answer is saved to
`output/<pet>/<file name>__<Gemini version>.json`, so a rerun of the timeline does not need new calls.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
record_files = sorted(p for p in RECORDS_DIR.iterdir() if p.suffix.lower() in MIME_TYPES)
print(f"{len(record_files)} record files in {RECORDS_DIR}")

rows = []
for path in record_files:
    for model in MODELS:
        extraction = extract_record(path, model)
        (OUTPUT_DIR / f"{path.stem}__{model}.json").write_text(
            extraction.model_dump_json(indent=2), encoding="utf-8"
        )
        for visit in extraction.visits:
            rows.append({"model": model, "file": path.name, **visit.model_dump()})
        print(f"{path.name} | {model} | {len(extraction.visits)} visits")

## The timeline

All visits from all files, oldest first, one table per Gemini version. Visits with no date are
listed last.

In [ ]:
timeline = pd.DataFrame(rows)
timeline["visit_date"] = pd.to_datetime(timeline["visit_date"], errors="coerce")
timeline = timeline.sort_values(["model", "visit_date", "file", "page"], na_position="last")

pd.set_option("display.max_colwidth", None)
for model in MODELS:
    print(model)
    display(timeline[timeline["model"] == model].drop(columns="model"))